# Fine-tuning an Encoder Model: Chinese Sentiment Classification

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/03-Finetune-Text-Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_errors.py', 'q_other_model.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"

## Dataset: ChnSentiCorp

Sentiment classification of Chinese reviews (hotels, books, electronics); `label` 1 is positive and 0 is negative. The data is in Chinese on purpose: this notebook is about fine-tuning a model for Chinese-language text, so the example inputs below are Chinese too.

`datasets` loads data in Apache Arrow format, which isn't read into memory all at once, so it can handle very large datasets.

In [ ]:
ds = load_dataset("lansinuote/ChnSentiCorp")
ds = ds.filter(lambda x: x["text"] is not None)
ds

In [ ]:
ds["train"][0]

In [ ]:
ds["train"].to_pandas().head(10)

In [ ]:
df = ds["train"].to_pandas()
print(df.label.value_counts())
df.text.str.len().describe()

## Model: mmBERT

Encoder models (the BERT family) are well suited to "understanding" tasks such as classification and retrieval, and they are much smaller and faster than LLMs.

[mmBERT](https://huggingface.co/jhu-clsp/mmBERT-small) (2025) is a multilingual model based on the ModernBERT architecture. It supports more than 1800 languages and sequences of up to 8192 tokens.

The sample sentence below means "The service at this hotel is very good."

In [ ]:
model_id = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer("這家飯店的服務很好")

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)
ds_tok = ds.map(tokenize, batched=True, remove_columns=["text"])
ds_tok

`AutoModelForSequenceClassification` adds a randomly initialized classification head on top of the pre-trained model.
(You'll see a warning that some weights are newly initialized. That's expected.)

In [ ]:
id2label = {0: "negative", 1: "positive"}
label2id = {v: k for k, v in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, id2label=id2label, label2id=label2id)
print(f"{model.num_parameters()/1e6:.0f}M parameters")

## Before Training: What Does the Model Do?

It takes token IDs as input and outputs logits for each class. The classification head is still random, so its predictions are just guesses.

The inputs mean "The service at this hotel is very good" and "The room is small and dirty".

In [ ]:
model.to(device)
inputs = tokenizer(["這家飯店的服務很好", "房間又小又髒"], padding=True, return_tensors="pt").to(device)
with torch.no_grad():
    print(model(**inputs).logits.softmax(-1))

## Fine-tuning with `Trainer`

`Trainer` takes care of the training loop, mixed precision, evaluation, checkpointing, logging, and more.
* `fp16=True`: use fp16 mixed precision on the T4
* `processing_class=tokenizer`: automatically uses `DataCollatorWithPadding` to pad each batch to the same length

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(-1) == labels).mean()}

args = TrainingArguments(
    output_dir="chnsenticorp-mmbert",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    num_train_epochs=2,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate(ds_tok["test"])

## Saving and Using the Model

Once saved to a folder, the model can be loaded with `from_pretrained` or `pipeline`, just like a model on the Hub.
You can also upload it to the Hugging Face Hub with `trainer.push_to_hub()` (requires logging in).

The test reviews mean "The service at this hotel is very good, and breakfast is plentiful", "The room is small and dirty, with poor soundproofing", and "It's cheap, but the location is a bit remote".

In [ ]:
trainer.save_model("my-sentiment-model")
clf = pipeline("text-classification", model="my-sentiment-model", device=device)
clf(["這家飯店的服務很好，早餐也很豐盛", "房間又小又髒，隔音很差", "價格便宜，但是位置有點偏僻"])

## Q
Find examples in the test set that the model gets wrong. What kinds of reviews are harder?

Hint: `trainer.predict(ds_tok["test"])`

In [ ]:
# %load q_errors.py
%run -i q_errors.py

## Q
Try other models and compare, for example:
* `google-bert/bert-base-chinese` (the classic Chinese BERT, 2018)
* `jhu-clsp/mmBERT-base` (a larger mmBERT)

Compare accuracy and training time.

In [ ]:
# %load q_other_model.py
%run -i q_other_model.py